# 3 2 count analysis he prior nmr

Peer-review copy. Inputs are pseudonymized processed data; see README.md and DATA_DICTIONARY.md. Original preparation/collection steps are outside this package.

In [ ]:
from review_setup import prepare_workspace
prepare_workspace()
from review_setup import initialize_stata
initialize_stata()


In [ ]:
import pandas as pd
from utils import get_source_corrections, StataDiD

source_corrections = get_source_corrections(window_days=28, horizon_days=28).query('prior_corrections==0').reset_index(drop=True)
source_corrections['prior_nmr'] = (source_corrections['nmr_count_28d']>1).astype(int)
source_corrections['prior_high_nmr'] = (source_corrections['nmr_high_score_count_28d']>0).astype(int)
print(source_corrections.shape)
source_corrections.head()

### Interaction analysis

In [ ]:
ts_posts = pd.read_parquet('data/ts_posts_weekly.parquet.gzip')
ts_posts = ts_posts.merge(source_corrections, on='tweet_id', how='inner')
print(ts_posts.shape)
ts_posts.head()

In [ ]:
stata_did = StataDiD(ts_posts, 'treated', 'tweet_id', 'weeks_from_display', -4, 4)

In [ ]:
from utils import get_coefs
stata_did.inter_did(reg='qui ppmlhdfe', dev='tweet_count', cluster='account_cluster', interv='prior_nmr')
means_mp_atts, lls_mp_atts, hls_mp_atts = get_coefs(reg='qui ppmlhdfe', start_line=4, end_line=6, stat_type='z', irr_or=1, unit_group=True)

### Subgroup

In [ ]:
stata_did.multi_did(reg='qui ppmlhdfe', dev='tweet_count', cluster='account_cluster', condition='if prior_nmr==1')
means_mp_atts, lls_mp_atts, hls_mp_atts = get_coefs(reg='qui ppmlhdfe', start_line=8, end_line=15, stat_type='z', irr_or=1, unit_group=True)

#### Honest DiD

In [ ]:
from pystata import stata
stata.run('matrix list e(b)')

In [ ]:
ylims = (-0.6, 0.7)
stata.run(f"local plotopts xtitle(Mbar, size(large)) ytitle(95% Robust CI, size(large)) yscale(r({ylims[0]} {ylims[1]})) ylabel({ylims[0]}(0.2){ylims[1]},labsize(large)) xlabel(,labsize(large)) ciopts(lwidth(thick)) xsize(4) ysize(3)")

for i in range(1,5):
    print(f"Post-treatment period {i}: \n")
    vec = [str(1) if e==i else str(0) for e in range(1,5)]
    vec_str = ' \\ '.join(vec)
    stata.run(f"matrix l_vec = {vec_str}")
    stata.run("honestdid, l_vec(l_vec) pre(9/11) post(12/15) mvec(0.5(0.5)2) alpha(0.05) grid_lb(-3) grid_ub(3)")
    stata.run("honestdid, cached coefplot `plotopts'")
    stata.run(f"graph export outputs/figures/honest_did_posts_weekly_nmr_{i}.pdf, replace")

In [ ]:
stata_did.multi_pre_did(reg='qui ppmlhdfe', dev='tweet_count', cluster='account_cluster', condition='if prior_nmr==1')

In [ ]:
stata.run('matrix list e(b)')

In [ ]:
print(f"Post-treatment period as a whole: \n")
vec = [str(1)]
vec_str = ' \\ '.join(vec)
stata.run(f"matrix l_vec = {vec_str}")
stata.run("honestdid, l_vec(l_vec) pre(9/11) post(12/12) mvec(0.5(0.5)2) alpha(0.05) grid_lb(-3) grid_ub(3)")
stata.run("honestdid, cached coefplot `plotopts'")
stata.run(f"graph export outputs/figures/honest_did_posts_weekly_nmr_agg.pdf, replace")

In [ ]:
stata_did.tw_did(reg='qui ppmlhdfe', dev='tweet_count', cluster='account_cluster', condition='if prior_nmr==1')
mean_agg_att, ll_agg_att, hl_agg_att = get_coefs(reg='qui ppmlhdfe', start_line=2, end_line=3, stat_type='z', irr_or=1, unit_group=True)

In [ ]:
from utils import att_plot
ylims = (-0.262, 0.462)
att_plot(-4, 4, 4, means_mp_atts, lls_mp_atts, hls_mp_atts, mean_agg_att, ll_agg_att, hl_agg_att, 'Weekly', 'Weeks from note display', 'Estimated DiD effect\n(repeated prior NMR note)', 'did_posts_weekly_nmr_1', ylims=ylims, legend_ncol=1, legend_title='First correction')

In [ ]:
stata_did.tw_did(reg='qui ppmlhdfe', dev='tweet_count', cluster='account_cluster', condition='if prior_nmr==0')
mean_agg_att, ll_agg_att, hl_agg_att = get_coefs(reg='qui ppmlhdfe', start_line=2, end_line=3, stat_type='z', irr_or=1, unit_group=True)

### Synthetic DiD

In [ ]:
from utils import SEED
from pystata import stata
stata.run(
    f"""
    sdid tweet_count unitId weeks_from_display treated_after if prior_nmr==1, ///
        vce(bootstrap) cluster(account_cluster) ///
        xline_opts(lcolor(none)) ///
        reps(100) seed({SEED}) graph ///
        g2_opt(xline(0, lcolor(black) lpattern(dash)) xtitle("Weeks from note display") ytitle("Post count") xlabel(-4(1)4) ylabel(60(5)100) title("") graphregion(color(white)))
    """
)

In [ ]:
stata.run(f"graph export outputs/figures/sdid_posts_weekly_nmr_1.pdf, replace")

### Sensitivity across horizons

In [ ]:
stata_did.tw_did(reg='qui ppmlhdfe', dev='tweet_count', cluster='account_cluster', condition='if days_from_rollout>60 & nmr_count_60d>1')
mean_agg_att_60, ll_agg_att_60, hl_agg_att_60 = get_coefs(reg='qui ppmlhdfe', start_line=2, end_line=3, stat_type='z', irr_or=1, unit_group=True)

In [ ]:
stata_did.tw_did(reg='qui ppmlhdfe', dev='tweet_count', cluster='account_cluster', condition='if days_from_rollout>90 & nmr_count_90d>1')
mean_agg_att_90, ll_agg_att_90, hl_agg_att_90 = get_coefs(reg='qui ppmlhdfe', start_line=2, end_line=3, stat_type='z', irr_or=1, unit_group=True)

In [ ]:
stata_did.tw_did(reg='qui ppmlhdfe', dev='tweet_count', cluster='account_cluster', condition='if days_from_rollout>180 & nmr_count_180d>1')
mean_agg_att_180, ll_agg_att_180, hl_agg_att_180 = get_coefs(reg='qui ppmlhdfe', start_line=2, end_line=3, stat_type='z', irr_or=1, unit_group=True)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from utils import FONT_SIZE, LW, MZ, vermilion

plt.style.use('default')
plt.rcParams['font.size'] = FONT_SIZE
plt.subplots(layout='constrained')
estimates = [mean_agg_att_60, mean_agg_att_90, mean_agg_att_180]
lls = [ll_agg_att_60, ll_agg_att_90, ll_agg_att_180]
hls = [hl_agg_att_60, hl_agg_att_90, hl_agg_att_180]
xticks = ['60', '90', '180']
plt.axhline(y=0, linestyle='--', color='lightgrey', linewidth=LW)
for i in range(len(estimates)):
    plt.errorbar(xticks[i], estimates[i][0], yerr=[[estimates[i][0]-lls[i][0]], [hls[i][0]-estimates[i][0]]],
                 marker='_', linestyle='', color=vermilion, markersize=MZ, mew=LW, linewidth=LW)
plt.xticks(np.arange(len(xticks)), xticks)
plt.xlim(-0.25, len(xticks)-0.75)
plt.ylim(ylims)
plt.xlabel('Pre-correction horizons (days)')
plt.ylabel('Estimated DiD effect\n(repeated prior NMR note)')
plt.savefig('outputs/figures/did_posts_weekly_nmr_horizons.pdf')
plt.show()

### Validation on high-score NMR note

In [ ]:
stata_did.multi_did(reg='qui ppmlhdfe', dev='tweet_count', cluster='account_cluster', condition='if prior_high_nmr==1')
means_mp_atts, lls_mp_atts, hls_mp_atts = get_coefs(reg='qui ppmlhdfe', start_line=8, end_line=15, stat_type='z', irr_or=1, unit_group=True)

In [ ]:
stata_did.tw_did(reg='qui ppmlhdfe', dev='tweet_count', cluster='account_cluster', condition='if prior_high_nmr==1')
mean_agg_att, ll_agg_att, hl_agg_att = get_coefs(reg='qui ppmlhdfe', start_line=2, end_line=3, stat_type='z', irr_or=1, unit_group=True)